# Held-out evaluation set for input validation

Builds the images the OOD evaluation harness needs, from the same NIH ChestX-ray14 mount the
calibration notebook uses.

Two directories come out of this:

- `positive/` — chest X-rays the CLIP prototype has **never seen**
- `negative/degraded_chest/` — those same studies deliberately corrupted

The exclusion is the whole point. `clip_prototype.json` was fitted on a specific draw from this
dataset; measuring acceptance on images inside that draw reports how well the threshold fits its
own training data, not how well it accepts unseen studies. That number would be wrong in the
flattering direction, which is the worst direction for it to be wrong in.

Run on Kaggle, download the zip at the end, unpack into `data/ood_eval/` in the repo.

## Import Library

In [ ]:
"""Build a held-out positive set and a degraded-negative set for validator evaluation.

The calibration draw is reproduced from the same constants and seed so it can be identified
and excluded — it was never recorded at the time, only re-derivable.

Companion to: notebooks/clip_calibration.ipynb
Consumed by:  scripts/eval_validator.py
"""

import json
import random
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from PIL import Image, ImageEnhance, ImageOps

## Config

In [ ]:
DATA_ROOT   = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")
CSV_PATH    = DATA_ROOT / "Data_Entry_2017.csv"
IMAGE_DIRS  = [DATA_ROOT / f"images_{i:03d}/images" for i in range(1, 13)]

OUT_ROOT    = Path("/kaggle/working/ood_eval")
POSITIVE_DIR = OUT_ROOT / "positive"
DEGRADED_DIR = OUT_ROOT / "negative" / "degraded_chest"
ZIP_PATH     = Path("/kaggle/working/ood_eval.zip")

# Must mirror clip_calibration.ipynb exactly, or the exclusion list is wrong 
CALIB_N_NO_FINDING    = 250
CALIB_N_PER_CONDITION = 18
CALIB_SEED            = 42

# Held-out sample 
# 250 keeps FPR@95TPR stable enough to compare two validators. That metric reads the 95th
# percentile of the positive scores, and extreme quantiles are noisy: at n=100 the estimate
# swings by roughly +/-19 points, at n=200 by +/-14, and past n=300 the gain flattens out.
N_POSITIVE = 250
HELD_SEED = 7    # deliberately not CALIB_SEED

# 5 transforms x 20 = 100 degraded negatives
N_PER_TRANSFORM = 20
DEGRADE_SEED = 42

CONDITIONS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Effusion", "Emphysema", "Fibrosis", "Hernia",
    "Infiltration", "Mass", "Nodule", "Pleural_Thickening",
    "Pneumonia", "Pneumothorax",
]

## Helper Function

In [ ]:
def build_image_index() -> dict:
    """Precompute filename to path mapping across all 12 image subdirectories."""
    index = {}
    for d in IMAGE_DIRS:
        if d.exists():
            for p in d.iterdir():
                index[p.name] = p
    return index


def calibration_filenames(df: pd.DataFrame) -> set:
    """Reproduce the calibration draw and return the distinct filenames it touched.

    The draw takes one sample per condition a study matches and never de-duplicates, so a
    multi-label study is pulled once for each of its findings. N_PER_CONDITION x 14 is
    therefore a count of draws, not of images; taking the set recovers what was really used.
    """
    used = []

    no_finding = df[df["Finding Labels"] == "No Finding"]
    used.extend(
        no_finding.sample(n=min(CALIB_N_NO_FINDING, len(no_finding)),
                          random_state=CALIB_SEED)["Image Index"]
    )

    for cond in CONDITIONS:
        positive = df[df["Finding Labels"].str.contains(cond, na=False)]
        n = min(CALIB_N_PER_CONDITION, len(positive))
        if n > 0:
            used.extend(positive.sample(n=n, random_state=CALIB_SEED)["Image Index"])

    print(f"Calibration draw : {len(used)} draws over {len(set(used))} distinct images")
    return set(used)


def sample_held_out(df: pd.DataFrame, excluded: set, n: int) -> list:
    """Draw n distinct filenames the calibration never saw, keeping the same stratum mix.

    The mix matters: acceptance measured on an all-"No Finding" sample would be easier than
    the population the threshold was fitted to, and measured on an all-pathology sample,
    harder. Either way the number would not describe normal use.
    """
    available = df[~df["Image Index"].isin(excluded)]
    rng = random.Random(HELD_SEED)

    half = n // 2
    no_finding = available[available["Finding Labels"] == "No Finding"]["Image Index"].tolist()
    picked = set(rng.sample(no_finding, min(half, len(no_finding))))

    per_condition = max((n - len(picked)) // len(CONDITIONS), 1)
    for cond in CONDITIONS:
        pool = available[
            available["Finding Labels"].str.contains(cond, na=False)
            & ~available["Image Index"].isin(picked)
        ]["Image Index"].tolist()
        if pool:
            picked.update(rng.sample(pool, min(per_condition, len(pool))))

    if len(picked) < n:
        remainder = [f for f in available["Image Index"].tolist() if f not in picked]
        picked.update(rng.sample(remainder, min(n - len(picked), len(remainder))))

    return sorted(picked)[:n]


def centre_crop(image: Image.Image, keep: float) -> Image.Image:
    """Crop to the central fraction, standing in for a badly collimated study."""
    w, h = image.size
    nw, nh = int(w * keep), int(h * keep)
    left, top = (w - nw) // 2, (h - nh) // 2
    return image.crop((left, top, left + nw, top + nh))


# Each transform names the real acquisition or handling fault it stands in for. All five stay
# recognisable as chest X-rays, so a prompt classifier passes every one of them — which is
# exactly why they belong in the set. Anything caught here was caught by distribution.
TRANSFORMS = {
    "inverted":      lambda im: ImageOps.invert(im.convert("L")).convert("RGB"),
    "hard_crop":     lambda im: centre_crop(im, 0.45),
    "rotated_90":    lambda im: im.rotate(90, expand=True),
    "flat_contrast": lambda im: ImageEnhance.Contrast(im).enhance(0.25),
    "blown_out":     lambda im: ImageEnhance.Brightness(im).enhance(2.1),
}

## Main Run — held-out positives

In [ ]:
df = pd.read_csv(CSV_PATH, usecols=["Image Index", "Finding Labels"])
image_index = build_image_index()
print(f"Image index      : {len(image_index)} files")

df = df[df["Image Index"].isin(image_index)]
excluded = calibration_filenames(df)

chosen = sample_held_out(df, excluded, N_POSITIVE)

overlap = set(chosen) & excluded
assert not overlap, f"Held-out sample overlaps calibration on {len(overlap)} images"
assert len(chosen) == len(set(chosen)), "Held-out sample contains duplicates"

POSITIVE_DIR.mkdir(parents=True, exist_ok=True)
for filename in chosen:
    shutil.copy2(image_index[filename], POSITIVE_DIR / filename)

no_finding_share = sum(
    1 for f in chosen
    if df.loc[df["Image Index"] == f, "Finding Labels"].iloc[0] == "No Finding"
) / len(chosen)

print(f"Held-out written : {len(chosen)} -> {POSITIVE_DIR}")
print(f"Overlap          : 0")
print(f"No Finding share : {no_finding_share:.0%}")

# Record what was taken, so a later run can exclude this set too if you ever grow it.
(OUT_ROOT / "positive_manifest.json").write_text(json.dumps(sorted(chosen), indent=2))

## Main Run — degraded negatives

Derived from the positives just written, so they inherit the same exclusion for free.

In [ ]:
sources = sorted(POSITIVE_DIR.iterdir())
rng = random.Random(DEGRADE_SEED)
DEGRADED_DIR.mkdir(parents=True, exist_ok=True)

written = 0
for name, transform in TRANSFORMS.items():
    for path in rng.sample(sources, min(N_PER_TRANSFORM, len(sources))):
        with Image.open(path) as image:
            transform(image.convert("RGB")).save(DEGRADED_DIR / f"{name}__{path.stem}.png")
        written += 1

print(f"Degraded written : {written} -> {DEGRADED_DIR}")
print("Strata           :", ", ".join(TRANSFORMS))

## Package for download

One archive is far less painful than pulling 350 files out of the Kaggle output browser.
Unpack it into `data/ood_eval/` in the repo.

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUT_ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(OUT_ROOT))

size_mb = ZIP_PATH.stat().st_size / 1e6
print(f"Archive : {ZIP_PATH}  ({size_mb:.1f} MB)")
print()
print("Next, locally:")
print("  unzip ood_eval.zip -d data/ood_eval/")
print("  python scripts/collect_ood_samples.py --source <download> --stratum other_radiograph --n 80")
print("  python scripts/eval_validator.py --csv runs/clip_baseline.csv")